# Agentic RL from Scratch: Teaching a Tiny LLM to Win Tic-Tac-Toe with GRPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ProKil/salt-lab-rl-tutorial/blob/main/agentic_rl_grpo_tictactoe.ipynb) *(use a T4 GPU runtime: Runtime → Change runtime type → T4 GPU)*

We take a small language model, [`Qwen/Qwen3.5-0.8B`](https://huggingface.co/Qwen/Qwen3.5-0.8B), drop it into a game environment, and train it with **GRPO** (Group Relative Policy Optimization) until it stops losing. The LLM *is* the policy: it reads a board as text and writes back a move. This is the core loop behind RL-trained reasoning and agent models, shrunk to something you can watch converge.

**The three moving parts**
- **Environment** — tic-tac-toe. Gives states (boards), accepts actions (moves), returns a reward at the end.
- **Policy** — the LLM $\pi_\theta$. Prompt in (a board), token sequence out (a move).
- **Algorithm** — GRPO. Sample a *group* of games, score them, and push the policy toward the above-average ones. No value network, no critic.

**A word on the "100% win rate" goal (read this).** Tic-tac-toe is a forced draw under optimal play, so no first player can *win* 100% of games against a competent opponent. What is achievable:
- vs a **weak, deterministic opponent** (the default here): literally **100% wins** are reachable, because the game becomes deterministic and exploitable. Great for watching the number hit 1.0.
- vs a **random opponent** (the harder variant): **0% losses** and a high win rate (~90%+) are reachable; the rest are unavoidable draws.

We frame success as *drive the loss rate to zero, then maximize wins*, and let you pick the opponent.

**Compute note.** Parts 1, 2, 3, 5, 6, 7 are pure Python/NumPy/PyTorch-CPU: their fill-ins and tests **run anywhere**, and this is where the real learning is. Parts 0, 4, 8, 9 touch the actual 0.8B model and **need a GPU** (a free Colab T4 is enough; generations are tiny). If you have no GPU, set `MODEL_ID` to a small text model in Part 0 to smoke-test the whole pipeline on CPU.

As before: `# ===== FILL IN THE BLANK =====` cells are yours. Run the test cell after each. `✓` = pass. Full solutions are in the appendix.

## Part 0 — Install and load the policy

`Qwen3.5-0.8B` is a recent multimodal checkpoint, so it needs an up-to-date `transformers`. We only ever feed it **text**, so a plain tokenizer and a text-only forward pass are all we use. The loader below tries the standard causal-LM class and falls back to the image-text-to-text class if needed; either way we call the model with `input_ids` and read `.logits`.

We keep **two** copies of the model: the trainable policy `model`, and a frozen `ref_model` used only for the KL penalty that keeps training from drifting too far.

In [1]:
# GPU recommended. On Colab: Runtime -> Change runtime type -> GPU.
# Qwen3.5 needs a very recent transformers:
#   pip install -U "transformers>=4.57" accelerate torch
# (if the model fails to load, install from source:
#   pip install "transformers @ git+https://github.com/huggingface/transformers.git")

import torch, torch.nn.functional as F
import random

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!pip install -U peft
!pip install -U "torchao>=0.16.0"
!pip install flash-linear-attention


MODEL_ID = "Qwen/Qwen3.5-0.8B"
# No GPU? Swap in a tiny text model to exercise the pipeline on CPU, e.g.:
# MODEL_ID = "Qwen/Qwen3-0.6B"    # or any small AutoModelForCausalLM checkpoint

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  =  torch.float32
print("device:", DEVICE, "| dtype:", DTYPE, "| model:", MODEL_ID)


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 703.0/703.0 kB 41.4 MB/s eta 0:00:00


   ━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [fla-core]

   ━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [fla-core]

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 2/3 [flash-linear-attention]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 2/3 [flash-linear-attention]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [flash-linear-attention]



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


device: cuda | dtype: torch.float32 | model: Qwen/Qwen3.5-0.8B


In [2]:
from transformers import AutoTokenizer
from peft import LoraConfig, get_peft_model

def load_policy(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    import transformers
    base, last = None, None
    for name in ("AutoModelForCausalLM", "AutoModelForImageTextToText"):
        try:
            base = getattr(transformers, name).from_pretrained(model_id, dtype=DTYPE); break
        except Exception as e:
            last = e
    if base is None: raise RuntimeError(last)
    return tok, base.to(DEVICE)

tokenizer, base_model = load_policy(MODEL_ID)

# Freeze the base; train only small LoRA adapters -> tiny optimizer state, no base grads.
base_model.config.use_cache = False          # needed for gradient checkpointing
base_model.gradient_checkpointing_enable()
base_model.enable_input_require_grads()

lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0,
                      target_modules=["q_proj","k_proj","v_proj","o_proj"],
                      task_type="CAUSAL_LM")
model = get_peft_model(base_model, lora_cfg)
model.train()
model.print_trainable_parameters()           # ~3M trainable, not 0.9B

# No ref_model copy: the reference policy is the base model with adapters OFF.

W0707 19:43:10.608000 5 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W0707 19:43:10.668000 5 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 753,474,368 || trainable%: 0.1435


## Part 1 — The environment

A board is a length-9 list, row-major, with `0` = empty, `1` = our agent (plays **X**, moves first), `2` = the opponent (**O**). Cells are numbered `0..8`.

You implement `check_winner`, the rules engine. Everything else (legal moves, applying a move, the two opponents) is provided.

In [3]:
LINES = [(0,1,2),(3,4,5),(6,7,8),   # rows
         (0,3,6),(1,4,7),(2,5,8),       # cols
         (0,4,8),(2,4,6)]               # diagonals

def legal_moves(board):
    return [i for i in range(9) if board[i] == 0]

def apply_move(board, pos, player):
    b = list(board); b[pos] = player; return b

def random_opponent(board, rng):
    """Picks uniformly among empty cells."""
    return rng.choice(legal_moves(board))

def weak_opponent(board, rng):
    """Deterministic and exploitable: always the lowest-numbered empty cell."""
    return legal_moves(board)[0]


def check_winner(board):
    """Return 1 if X has three in a row, 2 if O does, 0 if the board is full
    with no winner (draw), or None if the game is still ongoing.

    Hint: scan LINES; a line (a,b,c) wins for player p if
          board[a] == board[b] == board[c] == p (and p != 0).
          If no winner and no empty cells remain -> draw (0), else ongoing (None).
    """
    # ===== FILL IN THE BLANK =====
    raise NotImplementedError("Fill in the blank: implement the win/draw/ongoing check")
    # =============================

In [4]:
# --- TEST: check_winner (runs on CPU) ---
def _check(name, cond):
    print(("\u2713 " if cond else "\u2717 ") + name); assert cond, name

_check("row win for X",        check_winner([1,1,1, 0,0,0, 0,0,0]) == 1)
_check("column win for O",     check_winner([2,0,0, 2,0,0, 2,0,0]) == 2)
_check("diagonal win for X",   check_winner([1,0,0, 0,1,0, 0,0,1]) == 1)
_check("full board is a draw", check_winner([1,2,1, 1,2,2, 2,1,1]) == 0)
_check("empty board ongoing",  check_winner([0]*9) is None)
_check("partial board ongoing",check_winner([1,2,0, 0,0,0, 0,0,0]) is None)
print("Part 1 passed.")

✓ row win for X
✓ column win for O
✓ diagonal win for X
✓ full board is a draw
✓ empty board ongoing
✓ partial board ongoing
Part 1 passed.


## Part 2 — Board ⇄ text

The policy speaks tokens, so we render the board into a prompt and parse a move back out. `render_prompt` is provided. You implement `parse_move`: pull the model's chosen cell out of free-form text, returning `ILLEGAL` if the model names no legal cell.

Robust parsing matters: a model that emits an illegal or unparseable move should be *penalized*, and that penalty is what teaches it the output format in the first place.

**Parse the first legal *digit*, not the first integer.** The untrained 0.8B model glues its answer to trailing junk — a typical raw sample is `'20/\n# 20'`, which means "cell 2" followed by garbage. An integer regex reads that as `20` → illegal. And if nearly *every* move parses as illegal, every game in the group gets reward `-1`, the group advantage is exactly zero, and so is the gradient: training can never start. (Measured on a T4: integer-parse ≈ 12% legal moves from the untrained model; first-digit parse ≈ 88%.) Reading the first legal digit preserves the reward variance GRPO needs to bootstrap.

In [5]:
ILLEGAL = -1

def render_prompt(board):
    sym = {0:".", 1:"X", 2:"O"}
    rows = [" | ".join(sym[board[3*r+c]] for c in range(3)) for r in range(3)]
    grid = "\n-----\n".join(rows)
    empties = ", ".join(str(i) for i in legal_moves(board))
    user = (f"Tic-tac-toe. You are X. Cells are numbered 0-8.\n{grid}\n"
            f"Empty cells: {empties}\nReply with ONLY the cell number to play.")
    # Use the chat template so the model sees the format it was tuned on.
    msgs = [{"role":"user","content":user}]
    # enable_thinking=False: Qwen3.x reasons by default and would spend the whole
    # max_new_tokens budget inside <think>...</think>, never emitting a digit -> every
    # move parses as ILLEGAL. Forcing a direct answer is what makes moves legal at all.
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)

def parse_move(text, board):
    """Extract the agent's move from `text`.
    Return the FIRST digit character in `text` that names a legal empty cell.
    If no such digit appears, return ILLEGAL.

    (First DIGIT, not first integer: the untrained model emits e.g. "20/\\n# 20"
    meaning cell 2 + junk. Integer-parsing reads 20 -> ILLEGAL, and all-illegal
    groups have zero advantage -> zero gradient -> training never starts.)

    Hint: scan text char by char; if ch.isdigit() and int(ch) is in
          legal_moves(board), return int(ch).
    """
    legal = set(legal_moves(board))
    # ===== FILL IN THE BLANK =====
    raise NotImplementedError("Fill in the blank: return first legal digit, else ILLEGAL")
    # =============================

In [6]:
# --- TEST: parse_move (runs on CPU; does not call the model) ---
b = [1,0,2, 0,0,0, 0,0,0]           # legal empties: 1,3,4,5,6,7,8
_check("parses a clean move",        parse_move("3", b) == 3)
_check("parses move inside a phrase", parse_move("I'll take 4, that's best", b) == 4)
_check("reads first digit even when glued to junk", parse_move("40/\n# 20", b) == 4)
_check("skips occupied cell 0",      parse_move("play 0", b) == ILLEGAL)
_check("skips occupied then legal",   parse_move("not 2 but 5", b) == 5)
_check("garbage -> ILLEGAL",          parse_move("hmm let me think", b) == ILLEGAL)
print("Part 2 passed.")

✓ parses a clean move
✓ parses move inside a phrase
✓ reads first digit even when glued to junk
✓ skips occupied cell 0
✓ skips occupied then legal
✓ garbage -> ILLEGAL
Part 2 passed.


## Part 3 — The reward

GRPO is *outcome-supervised*: one scalar per game, applied to every move that game made. Keep it simple and sparse.

| outcome  | reward | why |
|----------|--------|-----|
| win      | `+1`   | the goal |
| draw     | `0`    | acceptable, not great |
| loss     | `-1`   | avoid |
| illegal  | `-1`   | an illegal move forfeits the game |

Fill in the mapping.

In [7]:
def game_reward(outcome):
    """Map an outcome string to a scalar reward.
    outcome is one of: "win", "draw", "loss", "illegal".
    """
    # ===== FILL IN THE BLANK =====
    raise NotImplementedError("Fill in the blank: return the reward for each outcome")
    # =============================

In [8]:
# --- TEST: game_reward (CPU) ---
_check("win  -> +1", game_reward("win") == 1.0)
_check("draw ->  0", game_reward("draw") == 0.0)
_check("loss -> -1", game_reward("loss") == -1.0)
_check("illegal -> -1", game_reward("illegal") == -1.0)
print("Part 3 passed.")

✓ win  -> +1
✓ draw ->  0
✓ loss -> -1
✓ illegal -> -1
Part 3 passed.


## Part 4 — Self-play rollouts

Now we let the policy play. `play_game` runs one full game: on the agent's turn we render the board, sample a completion from the LLM, parse the move, and apply it (an illegal move ends the game immediately). It records, for every agent turn, the exact token ids of `(prompt + response)` and where the prompt ends, so we can score those tokens later.

This part **calls the model**, so it needs a GPU (or a small `MODEL_ID`). The logic is provided. Read `sample_move` and `play_game` closely, they are the agent loop.

> The test cell here swaps the LLM for a random agent so you can verify the game loop itself on CPU.

In [9]:
GEN_KWARGS = dict(do_sample=True, temperature=1.0, top_p=1.0, top_k=20,
                  max_new_tokens=4)   # a move is ONE digit; a longer budget just buys
                                      # trailing junk ("0/\n# ...") after the answer

@torch.no_grad()
def sample_move(board):
    """Ask the policy for one move. Returns (pos, full_ids, prompt_len)."""
    prompt = render_prompt(board)
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    prompt_len = enc.input_ids.shape[1]
    out = model.generate(**enc, pad_token_id=tokenizer.pad_token_id, **GEN_KWARGS)
    full_ids = out[0]                                  # (prompt_len + response_len,)
    response = tokenizer.decode(full_ids[prompt_len:], skip_special_tokens=True)
    return parse_move(response, board), full_ids, prompt_len

def play_game(opponent_fn, rng, agent_fn=None):
    """Play one game, agent is X and moves first.
    Returns (turns, outcome) where turns is a list of dicts with token ids for
    each agent move. agent_fn overrides the LLM (used for CPU testing)."""
    board, turns = [0]*9, []
    while True:
        # --- agent (X) ---
        if agent_fn is None:
            pos, full_ids, prompt_len = sample_move(board)
            rec = {"full_ids": full_ids, "prompt_len": prompt_len}
        else:
            pos, rec = agent_fn(board), {}
        turns.append({**rec, "pos": pos})   # record the move BEFORE checking legality,
        if pos not in legal_moves(board):    # so an illegal move's tokens are in the
            return turns, "illegal"          # batch and get the -1 (this is what
                                             # actually teaches the output format)
        board = apply_move(board, pos, 1)
        w = check_winner(board)
        if w is not None:
            return turns, {1:"win", 0:"draw"}[w]
        # --- opponent (O) ---
        board = apply_move(board, opponent_fn(board, rng), 2)
        w = check_winner(board)
        if w is not None:
            return turns, {2:"loss", 0:"draw"}[w]

In [10]:
# --- TEST: game loop with a RANDOM agent (CPU, no model needed) ---

rng = random.Random(0)
rand_agent = lambda board: rng.choice(legal_moves(board))
counts = {"win":0,"draw":0,"loss":0,"illegal":0}
for _ in range(500):
    _, outcome = play_game(random_opponent, rng, agent_fn=rand_agent)
    counts[outcome] += 1
_check("games all terminate with valid outcomes", sum(counts.values()) == 500)
_check("random agent sometimes wins and sometimes loses", counts["win"]>0 and counts["loss"]>0)
print("outcome counts (random vs random):", counts)
print("Part 4 loop verified.")

✓ games all terminate with valid outcomes
✓ random agent sometimes wins and sometimes loses
outcome counts (random vs random): {'win': 289, 'draw': 73, 'loss': 138, 'illegal': 0}
Part 4 loop verified.


## Part 5 — GRPO advantages

Here is the one idea that makes GRPO *GRPO*. Play a **group** of $G$ games, collect their rewards $r_1,\dots,r_G$, and turn each reward into an advantage by normalizing **within the group**:
$$\hat A_i = \frac{r_i - \operatorname{mean}(r)}{\operatorname{std}(r) + \varepsilon}.$$

No critic, no value function. The group's own mean is the baseline. Above-average games get pushed up, below-average games get pushed down, and the scale is set by the group's spread. (If every game in the group ties, $\operatorname{std}=0$ and all advantages are ~0: nothing to learn from that group.)

In [11]:
def grpo_advantages(rewards, eps=1e-6):
    """Group-relative advantages.
    rewards : 1-D float tensor of shape (G,)
    returns : 1-D tensor (G,), the within-group standardized rewards.
    Hint: use rewards.mean() and rewards.std(unbiased=False); add eps to the std.
    """
    r = rewards.float()
    # ===== FILL IN THE BLANK =====
    raise NotImplementedError("Fill in the blank: (r - mean) / (std + eps)")
    # =============================

In [12]:
# --- TEST: grpo_advantages (CPU) ---
adv = grpo_advantages(torch.tensor([1., 1., -1., 0.]))
_check("advantages are ~zero-mean", abs(adv.mean().item()) < 1e-5)
_check("winning games rank above losing ones", adv[0] > adv[2])
_check("equal ranks map to equal advantages", torch.isclose(adv[0], adv[1]))
_check("degenerate group -> ~0 advantages",
       grpo_advantages(torch.tensor([1.,1.,1.])).abs().max() < 1e-3)
print("Part 5 passed.")

✓ advantages are ~zero-mean
✓ winning games rank above losing ones
✓ equal ranks map to equal advantages
✓ degenerate group -> ~0 advantages
Part 5 passed.


## Part 6 — Scoring tokens: log-probabilities

To do a policy-gradient update we need $\log \pi_\theta(\text{token})$ for each generated token. Two pieces:

1. `selected_logprobs(logits, targets)` — the pure part, and your job: given the model's `logits` (shape `(B, L, V)`) and the realized `targets` (shape `(B, L)`), return the log-probability the model assigned to each target token.
2. `compute_logprobs(a_model, seqs, prompt_lens)` — provided. It runs the forward pass, shifts logits so position $t$ predicts token $t{+}1$, calls your `selected_logprobs`, and builds a **mask** that is 1 only on the *response* tokens (never the prompt, never padding). We only train on tokens the policy actually generated.

In [13]:
def selected_logprobs(logits, targets):
    """Log-prob of each target token.
    logits  : (B, L, V) real-valued scores
    targets : (B, L) integer token ids
    returns : (B, L) with logits turned into log-probs and gathered at targets.
    Hint: F.log_softmax over the vocab dim, then .gather(-1, targets[...,None]).squeeze(-1)
    """
    # ===== FILL IN THE BLANK =====
    raise NotImplementedError("Fill in the blank: log_softmax then gather at targets")
    # =============================

In [14]:
def compute_logprobs(a_model, seqs, prompt_lens):
    """(Provided.) Per-token response log-probs and a response mask.
    seqs : (B, T) padded token ids ; prompt_lens : (B,) length of each prompt.
    Returns logprobs (B, T-1), mask (B, T-1) that is 1 only on response tokens."""
    attn = (seqs != tokenizer.pad_token_id).long()
    logits = a_model(input_ids=seqs, attention_mask=attn).logits  # (B, T, V)
    logits, targets = logits[:, :-1, :], seqs[:, 1:]              # predict t+1
    logprobs = selected_logprobs(logits, targets)                # (B, T-1)
    pos = torch.arange(seqs.shape[1] - 1, device=seqs.device)[None, :]
    resp_mask = (pos >= (prompt_lens[:, None] - 1)) & attn[:, 1:].bool()
    return logprobs, resp_mask.float()

In [15]:
# --- TEST: selected_logprobs against a manual computation (CPU) ---
torch.manual_seed(0)
logits = torch.randn(2, 3, 5)
targets = torch.tensor([[0, 1, 2], [4, 3, 1]])
got = selected_logprobs(logits, targets)
want = torch.stack([F.log_softmax(logits[i], -1)[torch.arange(3), targets[i]]
                    for i in range(2)])
_check("selected_logprobs shape (B, L)", got.shape == (2, 3))
_check("selected_logprobs matches manual gather", torch.allclose(got, want, atol=1e-6))
_check("log-probs are non-positive", bool((got <= 1e-4).all()))
print("Part 6 passed.")

✓ selected_logprobs shape (B, L)
✓ selected_logprobs matches manual gather
✓ log-probs are non-positive
Part 6 passed.


## Part 7 — The GRPO loss

This is the heart. For each response token we form the importance ratio against the sampling ("old") policy, apply the PPO-style clipped surrogate weighted by the game's advantage, and add a KL penalty pulling toward the frozen reference policy:

$$\mathcal{L} = -\underbrace{\min\!\Big(\rho_t \hat A,\; \operatorname{clip}(\rho_t, 1{-}\epsilon, 1{+}\epsilon)\,\hat A\Big)}_{\text{clipped surrogate}} \;+\; \beta\,\underbrace{\big(e^{\,\ell^{\text{ref}}_t-\ell_t} - (\ell^{\text{ref}}_t-\ell_t) - 1\big)}_{\text{KL}\ \ge\ 0},\qquad \rho_t = e^{\,\ell_t - \ell^{\text{old}}_t},$$

averaged over response tokens (using the mask). Here $\ell_t=\log\pi_\theta$, $\ell^{\text{old}}$ is detached (from sampling time), $\ell^{\text{ref}}$ is the frozen reference. The KL uses the standard unbiased $k_3$ estimator, which is always $\ge 0$.

With a single gradient step per batch, $\ell^{\text{old}}=\ell_t$ so $\rho_t=1$ and the surrogate is just $\hat A\cdot\ell_t$ — plain advantage-weighted REINFORCE with a group baseline. The clipping only bites once you take multiple inner steps on the same batch, but we write the full form so it generalizes.

In [16]:
def grpo_loss(new_logp, old_logp, ref_logp, advantages, mask,
              clip_eps=0.2, kl_coef=0.02):
    """GRPO per-token loss, averaged over response tokens.
    new_logp, old_logp, ref_logp : (B, L) log-probs (old & ref are detached)
    advantages : (B,) one scalar per sequence (broadcast across its tokens)
    mask       : (B, L) 1.0 on response tokens, else 0.0
    Returns a scalar loss to MINIMIZE.

    Steps:
      1. A = advantages broadcast to (B, 1)
      2. ratio = exp(new_logp - old_logp)
      3. surrogate = min(ratio * A, clip(ratio, 1-clip_eps, 1+clip_eps) * A)
      4. kl = exp(ref_logp - new_logp) - (ref_logp - new_logp) - 1     # >= 0
      5. per_token = -surrogate + kl_coef * kl
      6. return the mask-weighted mean of per_token
    """
    A = advantages.unsqueeze(-1)
    # ===== FILL IN THE BLANK =====
    raise NotImplementedError("Fill in the blank: clipped surrogate + KL, masked mean")
    # =============================

In [17]:
# --- TEST: grpo_loss properties (CPU) ---
B, L, V = 4, 6, 20
torch.manual_seed(1)
base = torch.randn(B, L, V)
tgt  = torch.randint(0, V, (B, L))
old  = selected_logprobs(base, tgt).detach()
ref  = old.clone()
mask = torch.ones(B, L); mask[:, 4:] = 0            # last 2 tokens are padding
adv  = grpo_advantages(torch.tensor([1., 1., -1., -1.]))

# on-policy first step: new == old  =>  ratio 1, kl 0  =>  loss = -mean(A over mask)
loss = grpo_loss(old.clone(), old, ref, adv, mask)
_check("loss is a finite scalar", loss.dim() == 0 and torch.isfinite(loss))

# gradient must push winning-game tokens UP and losing-game tokens DOWN
nl = old.clone().requires_grad_(True)
grpo_loss(nl, old, ref, adv, mask).backward()
g_win  = nl.grad[0, :4].mean().item()     # advantage > 0
g_lose = nl.grad[2, :4].mean().item()     # advantage < 0
_check("winning tokens pushed up (grad < 0)", g_win < 0)
_check("losing tokens pushed down (grad > 0)", g_lose > 0)
_check("masked (padding) tokens get no gradient", torch.allclose(nl.grad[:, 4:], torch.zeros(B,2)))

# KL is nonnegative when the policy moves away from the reference
drift = old + 0.5 * torch.randn(B, L)
kl = torch.exp(ref - drift) - (ref - drift) - 1.0
_check("KL estimator is >= 0", bool((kl >= -1e-6).all()))
print("Part 7 passed \u2014 the GRPO objective is correct.")

✓ loss is a finite scalar
✓ winning tokens pushed up (grad < 0)
✓ losing tokens pushed down (grad > 0)
✓ masked (padding) tokens get no gradient
✓ KL estimator is >= 0
Part 7 passed — the GRPO objective is correct.


## Part 8 — The training loop

Assemble everything. Each step: play `GROUP_SIZE` games, reward them, standardize to advantages, batch every agent turn, compute old/reference/new log-probs, take the GRPO loss, and step the optimizer. `inner_epochs > 1` reuses a batch (that is when clipping matters).

Provided end to end except the **optimizer step**, which is yours. This part **needs the model** (GPU, or a small `MODEL_ID`). Start with `SMOKE_TEST = True` to confirm the plumbing runs, then flip it off and let it cook.

In [18]:
from torch.nn.utils.rnn import pad_sequence

def collect_group(opponent_fn, rng, group_size):
    """Play a group of games; return (turns_all, advantages_per_turn, outcomes)."""
    games = [play_game(opponent_fn, rng) for _ in range(group_size)]
    rewards = torch.tensor([game_reward(o) for _, o in games])
    adv = grpo_advantages(rewards)
    turns, turn_adv = [], []
    for (game_turns, _), a in zip(games, adv):
        for t in game_turns:
            if "full_ids" in t:                 # skip illegal-first turns w/o tokens
                turns.append(t); turn_adv.append(a)
    return turns, turn_adv, [o for _, o in games]

def batch_turns(turns):
    seqs = pad_sequence([t["full_ids"] for t in turns],
                        batch_first=True, padding_value=tokenizer.pad_token_id)
    prompt_lens = torch.tensor([t["prompt_len"] for t in turns], device=seqs.device)
    return seqs.to(DEVICE), prompt_lens

def train_step(optimizer, opponent_fn, rng, group_size, inner_epochs=1,
               clip_eps=0.2, kl_coef=0.02):
    turns, turn_adv, outcomes = collect_group(opponent_fn, rng, group_size)
    if not turns:
        return 0.0, outcomes
    seqs, prompt_lens = batch_turns(turns)
    advantages = torch.stack(turn_adv).to(DEVICE)

    with torch.no_grad():
        old_logp, mask = compute_logprobs(model, seqs, prompt_lens)
        with model.disable_adapter():                 # frozen reference = base model
            ref_logp, _ = compute_logprobs(model, seqs, prompt_lens)

    last = 0.0
    for _ in range(inner_epochs):
        new_logp, _ = compute_logprobs(model, seqs, prompt_lens)
        loss = grpo_loss(new_logp, old_logp, ref_logp, advantages, mask,
                         clip_eps=clip_eps, kl_coef=kl_coef)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            (p for p in model.parameters() if p.requires_grad), 1.0)
        optimizer.step()
        last = loss.item()
    return last, outcomes

In [19]:
# --- Run training ---  (needs GPU or a small MODEL_ID)
SMOKE_TEST = True          # True: a few steps to prove it runs. False: real training.
OPPONENT   = weak_opponent # weak_opponent -> reachable 100% wins; random_opponent -> 0% losses

steps      = 5 if SMOKE_TEST else 400
GROUP_SIZE = 8      # 16 OOMs a 16GB T4 in float32 once games run full length
LR         = 1e-4   # raised: 1e-6 barely moves the adapters

import random

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
train_rng = random.Random(1234)
from collections import deque
recent = deque(maxlen=50)

for step in range(steps):
    loss, outcomes = train_step(optimizer, OPPONENT, train_rng, GROUP_SIZE)
    recent.extend(outcomes)
    win = recent.count("win")/max(len(recent),1)
    los = (recent.count("loss")+recent.count("illegal"))/max(len(recent),1)
    if step % (1 if SMOKE_TEST else 10) == 0:
        print(f"step {step:4d} | loss {loss:+.4f} | rolling win {win:.2f} loss {los:.2f}")
    if len(recent) == recent.maxlen and los == 0.0 and win >= 0.95:
        print(f"early stop at step {step}: rolling win {win:.2f} loss {los:.2f}")
        break
print("done.")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


[transformers] Caching is incompatible with gradient checkpointing in Qwen3_5DecoderLayer. Setting `past_key_values=None`.


/usr/local/lib/python3.12/site-packages/triton/language/core.py:2284: UserWarning: tl.make_block_ptr is deprecated. Use TensorDescriptor or tl.make_tensor_descriptor instead.
  warn("tl.make_block_ptr is deprecated. Use TensorDescriptor or tl.make_tensor_descriptor instead.")


step    0 | loss -0.0637 | rolling win 0.12 loss 0.75


step    1 | loss -0.2572 | rolling win 0.25 loss 0.69


step    2 | loss -0.0220 | rolling win 0.29 loss 0.62


step    3 | loss +0.0664 | rolling win 0.34 loss 0.59


step    4 | loss -0.0040 | rolling win 0.35 loss 0.60
done.


## Part 9 — Evaluation: did it learn to win?

Play a batch of games with **greedy** decoding against the chosen opponent and report the breakdown. This is the capstone check.

- vs `weak_opponent`: aim for **win rate → 1.00**.
- vs `random_opponent`: aim for **loss rate → 0.00** with wins as high as they go (draws are the ceiling on the rest).

In [20]:
@torch.no_grad()
def greedy_move(board):
    prompt = render_prompt(board)
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(**enc, do_sample=False, max_new_tokens=4,
                         pad_token_id=tokenizer.pad_token_id)
    resp = tokenizer.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)
    return parse_move(resp, board)

def evaluate(opponent_fn, n_games=100, seed=7):
    rng = random.Random(seed)
    tally = {"win":0,"draw":0,"loss":0,"illegal":0}
    for _ in range(n_games):
        _, outcome = play_game(opponent_fn, rng, agent_fn=greedy_move)
        tally[outcome] += 1
    return tally

tally = evaluate(OPPONENT, n_games=100)
win_rate  = tally["win"]/100
loss_rate = (tally["loss"]+tally["illegal"])/100
print("results over 100 games:", tally)
print(f"win rate: {win_rate:.2f} | loss rate: {loss_rate:.2f}")

# Soft capstone check (tighten as your model improves):
if OPPONENT is weak_opponent:
    print("✓ target met" if win_rate >= 0.95 else "keep training: aiming for win rate >= 0.95")
else:
    print("✓ target met" if loss_rate <= 0.02 else "keep training: aiming for loss rate <= 0.02")

results over 100 games: {'win': 100, 'draw': 0, 'loss': 0, 'illegal': 0}
win rate: 1.00 | loss rate: 0.00
✓ target met


## Where to go next

- **Harder opponents.** Swap in `random_opponent`, then a minimax opponent. Against optimal play the best attainable is a 100% *draw* rate: watch the reward structure force that.
- **Reward shaping.** Add a small bonus for well-formed output, or for taking center/forks, and see how it changes the learned style. Beware reward hacking.
- **Real GRPO scale.** The only things that change going from tic-tac-toe to math or coding are the environment and the reward function. The advantage, log-prob, and loss code you wrote here is the same. Frameworks like TRL (`GRPOTrainer`) and verl implement exactly this loop with batching, vLLM sampling, and multi-GPU.
- **KL and clipping.** Sweep `kl_coef` and `inner_epochs`. With `inner_epochs=1` the clip never triggers; push it up and watch the ratio move.

---
## Appendix — Solution key

Try first. If stuck, run this cell to define the reference implementations, then re-run the test cells above.

In [21]:
# ================= SOLUTION KEY (peek only if stuck) =================
def check_winner(board):
    for a,b,c in LINES:
        if board[a] != 0 and board[a] == board[b] == board[c]:
            return board[a]
    return 0 if all(v != 0 for v in board) else None

ILLEGAL = -1

def parse_move(text, board):
    # First legal DIGIT, not first integer: the untrained model glues its answer
    # to junk ("20/\n# 20" = cell 2 + garbage), and all-illegal groups have zero
    # advantage -> zero gradient -> training never starts.
    legal = set(legal_moves(board))
    for ch in text:
        if ch.isdigit() and int(ch) in legal:
            return int(ch)
    return ILLEGAL

def render_prompt(board):
    sym = {0:".", 1:"X", 2:"O"}
    rows = [" | ".join(sym[board[3*r+c]] for c in range(3)) for r in range(3)]
    grid = "\n-----\n".join(rows)
    empties = ", ".join(str(i) for i in legal_moves(board))
    user = (f"Tic-tac-toe. You are X. Cells are numbered 0-8.\n{grid}\n"
            f"Empty cells: {empties}\nReply with ONLY the cell number to play.")
    # Use the chat template so the model sees the format it was tuned on.
    msgs = [{"role":"user","content":user}]
    # enable_thinking=False: Qwen3.x reasons by default and would spend the whole
    # max_new_tokens budget inside <think>...</think>, never emitting a digit -> every
    # move parses as ILLEGAL. Forcing a direct answer is what makes moves legal at all.
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)

def game_reward(outcome):
    return {"win":1.0, "draw":0.0, "loss":-1.0, "illegal":-1.0}[outcome]

def grpo_advantages(rewards, eps=1e-6):
    r = rewards.float()
    return (r - r.mean()) / (r.std(unbiased=False) + eps)

def selected_logprobs(logits, targets):
    return F.log_softmax(logits, dim=-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)

def grpo_loss(new_logp, old_logp, ref_logp, advantages, mask,
              clip_eps=0.2, kl_coef=0.02):
    A = advantages.unsqueeze(-1)
    ratio = torch.exp(new_logp - old_logp)
    surrogate = torch.min(ratio * A, torch.clamp(ratio, 1-clip_eps, 1+clip_eps) * A)
    kl = torch.exp(ref_logp - new_logp) - (ref_logp - new_logp) - 1.0
    per_token = -surrogate + kl_coef * kl
    return (per_token * mask).sum() / mask.sum().clamp(min=1.0)

# The Part 8 optimizer-step blank is simply:
#     optimizer.zero_grad()
#     loss.backward()
#     torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#     optimizer.step()
print("Reference solutions loaded. Re-run the test cells to verify.")

Reference solutions loaded. Re-run the test cells to verify.
